# Importing libraries

In [74]:
import pandas as pd
import numpy as np
import yfinance as yf

# Importing data from Yahoo Finance

In [3]:
# Getting data for a single stock (e.g., Apple)
apple_data = yf.download("AAPL", start="2020-01-01", end="2020-01-15") 


# Geting data for multiple stocks
multi_data = yf.download(["AAPL", "MSFT"], start="2020-01-01", end="2020-01-15") 


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  2 of 2 completed


In [9]:
# Testing 5-year data download
apple_data_5y = yf.download("AAPL", start="2020-01-01", end="2025-02-06") 

[*********************100%***********************]  1 of 1 completed


In [10]:
apple_data_5y

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.796021,72.856613,71.545387,71.799873,135480400
2020-01-03,72.088295,72.851761,71.862892,72.020432,146322800
2020-01-06,72.662704,72.701485,70.953995,71.206062,118387200
2020-01-07,72.320992,72.929337,72.100433,72.672425,108872000
2020-01-08,73.484360,73.787323,72.022865,72.022865,132079200
...,...,...,...,...,...
2025-01-30,237.589996,240.789993,237.210007,238.669998,55658300
2025-01-31,236.000000,247.190002,233.440002,247.190002,101075100


In [11]:
# List of stocks
# https://stockanalysis.com/stocks/

In [71]:
# Getting cleaner data after dropping level:
apple_data_5y_cleaned = apple_data_5y.droplevel('Ticker', axis=1)

In [24]:
apple_data_5y_cleaned = apple_data_5y.droplevel('Price', axis=1)

In [72]:
apple_data_5y_cleaned

Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,72.796021,72.856613,71.545387,71.799873,135480400
2020-01-03,72.088295,72.851761,71.862892,72.020432,146322800
2020-01-06,72.662704,72.701485,70.953995,71.206062,118387200
2020-01-07,72.320992,72.929337,72.100433,72.672425,108872000
2020-01-08,73.484360,73.787323,72.022865,72.022865,132079200
...,...,...,...,...,...
2025-01-30,237.589996,240.789993,237.210007,238.669998,55658300
2025-01-31,236.000000,247.190002,233.440002,247.190002,101075100
2025-02-03,228.009995,231.830002,225.699997,229.990005,73063300


# Using stocks in the S&P500

In [73]:
# GICS Sector, Sub-Industry, HQ location, etc.
pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [81]:
# Storing just the symbols into a list
tickers = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0].Symbol

# Including the changed tickers as well to remove survivorship bias
tickers_changed = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[1]

# Changing the date to datetime
start_date_analyzed = '2023-05-05'

# Getting only the added and removed tickers
tickers_added = tickers_changed[pd.to_datetime(tickers_changed.Date.Date) >= start_date_analyzed].Added
tickers_removed = tickers_changed[pd.to_datetime(tickers_changed.Date.Date) >= start_date_analyzed].Removed

# Removing the additional added tickers
tickers = tickers[~(tickers.isin(tickers_added.Ticker))]

# Appending the removed tickers
tickers = pd.concat([tickers, tickers_removed.Ticker], ignore_index=True)

# Dropping any NA values
tickers = tickers.dropna()

# Dropping duplicates
tickers = tickers.drop_duplicates()

# Resetting the index
tickers = tickers.reset_index(drop = True)

In [82]:
tickers

0       MMM
1       AOS
2       ABT
3      ABBV
4       ACN
       ... 
499     DXC
500     LNC
501     NWL
502     AAP
503    DISH
Length: 504, dtype: object

In [86]:
# Later Testing:

In [83]:
from datetime import date
pandemic_date = "2023-05-05"
today = date.today()

problematic_tickers = ['BF.B', 'BRK.B', 'GEV', 'SOLV', 'SW', 'VLTO']
filtered_tickers = [ticker for ticker in tickers if ticker not in problematic_tickers]

# Get Data for S&P500 Stocks
multi_data = yf.download(filtered_tickers, start=pandemic_date, end=today)

[*********************100%***********************]  501 of 501 completed

5 Failed downloads:
['PXD', 'MRO', 'DISH', 'ATVI']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
['LEN']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2023-05-05 -> 2025-02-13)')


In [85]:
multi_data

Price      Adj Close                        Close                         \
Ticker          ATVI DISH LEN MRO PXD           A        AAL         AAP   
Date                                                                       
2023-05-05       NaN  NaN NaN NaN NaN  131.640610  13.870000  119.869606   
2023-05-08       NaN  NaN NaN NaN NaN  130.801193  14.360000  121.787209   
2023-05-09       NaN  NaN NaN NaN NaN  125.764694  14.470000  121.564468   
2023-05-10       NaN  NaN NaN NaN NaN  126.465851  14.340000  121.835655   
2023-05-11       NaN  NaN NaN NaN NaN  126.070831  14.070000  120.257004   
...              ...  ...  ..  ..  ..         ...        ...         ...   
2025-02-06       NaN  NaN NaN NaN NaN  147.639999  17.059999   48.880001   
2025-02-07       NaN  NaN NaN NaN NaN  145.210007  17.170000   45.950001   
2025-02-10       NaN  NaN NaN NaN NaN  143.690002  16.620001   45.240002   
2025-02-11       NaN  NaN NaN NaN NaN  141.750000  16.410000   47.000000   
2025-02-12       NaN  NaN NaN NaN NaN  138.080002  16.260000   47.490002   

Price                               ...   Volume                              \
Ticker            AAPL        ABBV  ...     WYNN      XEL       XOM     XRAY   
Date                                ...                                        
2023-05-05  171.837067  138.537552  ...  1560300  2147900  14964700  1419400   
2023-05-08  171.767761  137.994736  ...  1836700  2171000  11187800   974400   
2023-05-09  170.055038  137.414474  ...  3359700  2102800  11370800  1670900   
2023-05-10  171.827164  137.030792  ...  5041500  2963100  14884800  1027000   
2023-05-11  172.015274  137.189865  ...  4509700  2850100  17165900   932600   
...                ...         ...  ...      ...      ...       ...      ...   
2025-02-06  232.963867  192.970001  ...  2458500  5611500  13411400  2181600   
2025-02-07  227.380005  190.600006  ...  2255300  3797800  10051600  1455000   
2025-02-10  227.649994  190.339996  ...  2620400  2638300  12142200  2115400   
2025-02-11  232.619995  191.830002  ...  2137600  2284300  12713100  1524200   
2025-02-12  236.869995  193.000000  ...  2755800  2706200  18240100  3503200   

Price                                                              
Ticker          XYL      YUM      ZBH     ZBRA      ZION      ZTS  
Date                                                               
2023-05-05  2884400  1361600   993900   486300  14680200  4224400  
2023-05-08  1523300   981400  1304500   384100  12038700  1976100  
2023-05-09  1452700  1094900  1017600   432900   4999300  1904200  
2023-05-10  1344700  1595200  1163400   257400   5689100  1610500  
2023-05-11  1278700  1144100   899700   349400   7247700  1539100  
...             ...      ...      ...      ...       ...      ...  
2025-02-06  1748600  5571800  3791900   579200   1170300  2419500  
2025-02-07  2040200  2694300  2803600  1423700    907300  1927100  
2025-02-10  1490400  3733500  2528700   859700    986500  2936100  
2025-02-11  1211900  2541800  2101800   951800   1031300  2309400  
2025-02-12  1718000  2167100  2496700  1316500    853800  2885600  

[445 rows x 2510 columns]

# Miscellaneous Testing